# common/models.ipynb — Khởi tạo mô hình

Notebook chứa hàm dùng chung, không tự train và không tự ghi kết quả. Các notebook chính nạp tự động bằng `%run`. Khi sửa hàm, khởi động lại kernel rồi Run All notebook chính.


## Nạp các hàm phụ thuộc
Cell này cũng cho phép mở và Run All riêng notebook dùng chung.

In [ ]:
from pathlib import Path
import os
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'common/data_io.ipynb').is_file()), None)
if root is None:
    raise FileNotFoundError('Mở notebook trong thư mục project đầy đủ.')
os.chdir(root)
get_ipython().run_line_magic('run', 'common/data_io.ipynb')
get_ipython().run_line_magic('run', 'common/preprocessing.ipynb')


## Thư viện và hằng số

In [ ]:
import numpy as np

from sklearn.base import BaseEstimator, ClassifierMixin

from sklearn.dummy import DummyClassifier

from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.pipeline import Pipeline


## `TorchMLP`
MLP PyTorch mẫu; tự chọn CUDA nếu có, lưu mô hình về CPU.

In [ ]:
class TorchMLP(ClassifierMixin, BaseEstimator):
    def __init__(self, hidden=64, epochs=80, learning_rate=0.001, seed=42):
        self.hidden, self.epochs, self.learning_rate, self.seed = hidden, epochs, learning_rate, seed

    def fit(self, X, y):
        try:
            import torch
            from torch import nn
        except ImportError as exc:
            raise ImportError('Install PyTorch to run the MLP: pip install torch') from exc
        torch.manual_seed(self.seed)
        self.classes_ = np.array(LABELS)
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.training_device_ = str(device)
        x = torch.as_tensor(np.asarray(X), dtype=torch.float32, device=device)
        targets = torch.tensor([LABELS.index(v) for v in y], dtype=torch.long, device=device)
        self.model_ = nn.Sequential(nn.Linear(x.shape[1], self.hidden), nn.ReLU(), nn.Linear(self.hidden, len(LABELS))).to(device)
        optimizer = torch.optim.Adam(self.model_.parameters(), lr=self.learning_rate)
        criterion = nn.CrossEntropyLoss()
        self.model_.train()
        for _ in range(self.epochs):
            optimizer.zero_grad()
            loss = criterion(self.model_(x), targets)
            loss.backward()
            optimizer.step()
        # Keep inference and the saved checkpoint portable to CPU machines.
        self.model_.to('cpu')
        return self

    def predict_proba(self, X):
        import torch
        self.model_.eval()
        with torch.no_grad():
            return torch.softmax(self.model_(torch.as_tensor(np.asarray(X), dtype=torch.float32)), dim=1).numpy()

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]


## `build_model`
Khởi tạo estimator/pipeline và lưới tham số Logistic, Random Forest. MLP giữ cấu hình mẫu.

In [ ]:
def build_model(name, X, seed):
    if name == 'dummy':
        return DummyClassifier(strategy='most_frequent'), {}
    prep = make_preprocessor(X)
    if name == 'logistic':
        clf = LogisticRegression(max_iter=2000, random_state=seed)
        grid = {'model__C': [0.1, 1.0, 10.0], 'model__class_weight': [None, 'balanced']}
    elif name == 'random_forest':
        clf = RandomForestClassifier(random_state=seed, n_jobs=1)
        grid = {'model__n_estimators': [100, 200], 'model__max_depth': [None, 10], 'model__class_weight': [None, 'balanced']}
    elif name == 'mlp':
        clf = TorchMLP(seed=seed)
        grid = {}
    else:
        raise ValueError(f'Unknown model: {name}')
    return Pipeline([('preprocess', prep), ('model', clf)]), grid
